# Learned Spectral Correction

A small neural network corrects the per-mode amplitudes of a spectrally truncated gyrokinetic solver to match the transport of a full-resolution reference. The corrector sits **inside** the simulation loop: after each solver block it rescales $\delta f$ per $k_y$, and the corrected state feeds back into the next block.

**Comparison**: fine solver (nky=32, nkx=85) vs coarse solver (nky=8, nkx=31, same dt) vs coarse + learned correction. Loss on heat flux.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import logging

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)

import sys

sys.path.append("..")

from gyaradax.bootstrap import init_jax

init_jax(device=5)

In [ ]:
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
import equinox as eqx
from tqdm.auto import tqdm

from gyaradax import compute_geometry, gksolve
from gyaradax.solver import linear_precompute
from gyaradax.simulate import gk_init as gk_init_fn
from gyaradax.plot_utils import JAX_COLORS

jax.config.update("jax_enable_x64", True)

## Setup

Load the iteration\_13 configuration. Fine solver at full resolution (nky=32, nkx=85), coarse with truncated spectrum (nky=16, nkx=43, ~4x fewer modes). Same dt — no CFL issues.

**Delete `figs/correction_cache.pkl` if grid sizes changed.**

In [ ]:
from gyaradax.params import load_config, gkparams_from_config
from gyaradax.utils import load_geometry

cfg = load_config("../configs/iteration_13.yaml")
data_dir = cfg.run.data_dir

# fine: full resolution from GKW files
geometry_fine = load_geometry(data_dir)

COARSE_NKY = 16
COARSE_NKX = 43
geometry_coarse = compute_geometry(
    q=float(cfg.geometry.q),
    shat=float(cfg.geometry.shat),
    eps=float(cfg.geometry.eps),
    ns=int(cfg.grid.ns),
    nkx=COARSE_NKX,
    nky=COARSE_NKY,
    nvpar=int(cfg.grid.nvpar),
    nmu=int(cfg.grid.nmu),
    vpar_max=float(cfg.grid.vpar_max),
    nperiod=int(cfg.grid.nperiod),
    krhomax=float(cfg.grid.krhomax),
    kxmax=float(cfg.geometry.kxmax),
    ikxspace=int(cfg.grid.ikxspace),
)

ky_fine = np.asarray(geometry_fine["krho"])
ky_coarse = np.asarray(geometry_coarse["krho"])

# same dt — coarsening is spectral only
DT = float(cfg.solver.dt)
NAVG = int(cfg.solver.naverage)
STEPS_PER_BLOCK = NAVG
BLOCK_TIME = STEPS_PER_BLOCK * DT

T_FINAL = 50.0
N_BLOCKS = int(T_FINAL / BLOCK_TIME)

params_fine = gkparams_from_config(cfg)
params_coarse = gkparams_from_config(cfg)

fine_modes = len(geometry_fine["kxrh"]) * len(ky_fine)
coarse_modes = COARSE_NKX * COARSE_NKY
print(f"fine:   nkx={len(geometry_fine['kxrh'])}, nky={len(ky_fine)} ({fine_modes} modes)")
print(
    f"coarse: nkx={COARSE_NKX}, nky={COARSE_NKY} ({coarse_modes} modes, {fine_modes/coarse_modes:.1f}x fewer)"
)
print(
    f"kx_coarse range: [{float(geometry_coarse['kxrh'][0]):.3f}, {float(geometry_coarse['kxrh'][-1]):.3f}]"
)
print(
    f"dt={DT}, {N_BLOCKS} blocks x {STEPS_PER_BLOCK} steps, total time={N_BLOCKS * BLOCK_TIME:.0f}"
)

## Fine reference run

Run the fine solver (full resolution, nky=32) from K01, collecting flux traces at each block.

In [ ]:
cache_path = "/tmp/correction_cache.pkl"

if os.path.exists(cache_path):
    print(f"loading cached results from {cache_path}...")
    with open(cache_path, "rb") as f:
        cache = pickle.load(f)
    fine_fluxes = cache["fine_fluxes"]
    fine_times = cache["fine_times"]
    coarse_fluxes = cache["coarse_fluxes"]
    coarse_times = cache["coarse_times"]
    coarse_dfs = cache["coarse_dfs"]
    coarse_states = cache["coarse_states"]
    print(f"loaded. fine: {len(fine_times)} blocks, coarse: {len(coarse_times)} blocks")
else:
    print("no cache found, running simulations...")

    # fine solver
    pre_fine = linear_precompute(geometry_fine, params_fine)
    df_fine, _, state_fine = gk_init_fn(geometry_fine, params_fine)
    fine_fluxes, fine_times = [], []

    for b in tqdm(range(N_BLOCKS), desc="fine"):
        df_fine, (phi, fluxes), state_fine = gksolve(
            df_fine,
            geometry_fine,
            params_fine,
            state_fine,
            n_steps=STEPS_PER_BLOCK,
            pre=pre_fine,
        )
        fine_fluxes.append([float(f) for f in fluxes])
        fine_times.append(float(state_fine.time))

    fine_fluxes = np.array(fine_fluxes)
    fine_times = np.array(fine_times)

    # coarse solver
    pre_coarse_ = linear_precompute(geometry_coarse, params_coarse)
    df_coarse, _, state_coarse = gk_init_fn(geometry_coarse, params_coarse)
    coarse_fluxes, coarse_times = [], []
    coarse_dfs, coarse_states = [], []
    coarse_dfs.append(df_coarse)
    coarse_states.append(state_coarse)

    for b in tqdm(range(N_BLOCKS), desc="coarse"):
        df_coarse, (phi, fluxes), state_coarse = gksolve(
            df_coarse,
            geometry_coarse,
            params_coarse,
            state_coarse,
            n_steps=STEPS_PER_BLOCK,
            pre=pre_coarse_,
        )
        coarse_fluxes.append([float(f) for f in fluxes])
        coarse_times.append(float(state_coarse.time))
        coarse_dfs.append(df_coarse)
        coarse_states.append(state_coarse)

    coarse_fluxes = np.array(coarse_fluxes)
    coarse_times = np.array(coarse_times)
    coarse_dfs = jnp.stack(coarse_dfs, axis=0)

    # save
    with open(cache_path, "wb") as f:
        pickle.dump(
            {
                "fine_fluxes": fine_fluxes,
                "fine_times": fine_times,
                "coarse_fluxes": coarse_fluxes,
                "coarse_times": coarse_times,
                "coarse_dfs": coarse_dfs,
                "coarse_states": coarse_states,
            },
            f,
        )
    print(f"saved to {cache_path}")

print(f"fine eflux:   [{fine_fluxes[:,1].min():.4f}, {fine_fluxes[:,1].max():.4f}]")
print(f"coarse eflux: [{coarse_fluxes[:,1].min():.4f}, {coarse_fluxes[:,1].max():.4f}]")

In [ ]:
# fine vs coarse flux comparison
fig, ax = plt.subplots(figsize=(7.0, 2.8))
ax.plot(
    fine_times,
    fine_fluxes[:, 1],
    color=JAX_COLORS["blue"],
    label=f"fine ($N_{{ky}}$={len(ky_fine)})",
)
ax.plot(
    coarse_times,
    coarse_fluxes[:, 1],
    color=JAX_COLORS["red"],
    ls="--",
    alpha=0.7,
    label=f"coarse ($N_{{ky}}$={COARSE_NKY})",
)
ax.set_xlabel(r"time $[v_{th}/R]$")
ax.set_ylabel(r"$Q_i$")
ax.legend(frameon=False)
ax.grid(True)
fig.tight_layout()
plt.show()

## Correction network

The MLP reads the coarse $k_y$ spectral energy and outputs a per-mode multiplicative correction. The corrected $\delta f$ feeds back into the next block. Loss: heat flux mismatch against the fine reference.

In [ ]:
pre_coarse = linear_precompute(geometry_coarse, params_coarse)


class ModeCorrector(eqx.Module):
    layers: list

    def __init__(self, n_ky, hidden, key=jax.random.PRNGKey(0)):
        k1, k2, k3 = jax.random.split(key, 3)

        self.layers = [
            eqx.nn.Linear(n_ky, hidden, key=k1),
            eqx.nn.Linear(hidden, hidden, key=k2),
            eqx.nn.Linear(hidden, n_ky, key=k3),
        ]

    def __call__(self, ky_energy):
        x = ky_energy
        for layer in self.layers[:-1]:
            x = jax.nn.silu(layer(x))
        x = self.layers[-1](x)
        # ky_shift, ky_scale = jnp.split(x, 2)
        # return ky_shift, ky_scale
        return 1.0 + jax.nn.sigmoid(x)


network = ModeCorrector(COARSE_NKY, hidden=64, key=jax.random.PRNGKey(42))
print(f"network params: {sum(x.size for x in jax.tree.leaves(network))}")

## Training

Loss: MSE on heat flux trace between corrected coarse and fine reference. Backprop through the full autoregressive chain.

In [ ]:
from gyaradax.integrals import get_integrals

UNROLL_BLOCKS = 1
# drop remainder
N_TRAIN = (N_BLOCKS // UNROLL_BLOCKS) * UNROLL_BLOCKS
target_fluxes = jnp.array(fine_fluxes[:N_TRAIN, 1])
target_chunks = target_fluxes.reshape(-1, UNROLL_BLOCKS)  # (num_chunks, UNROLL_BLOCKS)


@eqx.filter_jit
def train_batch(network, opt_state, carry_batch, targets_batch):
    """Unrolls solver for UNROLL_BLOCKS across a BATCH, computes grads, and returns state."""

    def batch_loss(net, carry_b, targets_b):
        # chunk_loss calculates the unroll for a single sequence
        def chunk_loss(net_inner, carry, targets):
            def step(c, target_flux):
                df, state = c
                df_next, (phi, _), state_next = gksolve(
                    df,
                    geometry_coarse,
                    params_coarse,
                    state,
                    n_steps=STEPS_PER_BLOCK,
                    pre=pre_coarse,
                )
                ky_energy = jnp.sum(phi.real**2 + phi.imag**2, axis=(0, 1))
                # ky_shift, ky_scale = net_inner(ky_energy)
                # ky_shift = ky_shift[None, None, None, None, :]
                # ky_scale = ky_scale[None, None, None, None, :]
                # df_corr = df_next * ky_scale + ky_scale
                correction = net_inner(ky_energy)
                df_corr = df_next * correction[None, None, None, None, :]

                _, fluxes_corr = get_integrals(
                    df_corr,
                    geometry_coarse,
                    params=params_coarse,
                    pre=pre_coarse,
                    adiabatic_electrons=params_coarse.adiabatic_electrons,
                )

                loss_step = (fluxes_corr[1] - target_flux) ** 2 / target_flux**2
                return (df_corr, state_next), loss_step

            final_carry, step_losses = jax.lax.scan(step, carry, targets)
            return jnp.mean(step_losses), final_carry

        # vmap over batch
        chunk_losses, final_carries = jax.vmap(chunk_loss, in_axes=(None, 0, 0))(
            net, carry_b, targets_b
        )
        return jnp.mean(chunk_losses), final_carries

    (loss, carry_out), grads = eqx.filter_value_and_grad(batch_loss, has_aux=True)(
        network, carry_batch, targets_batch
    )
    updates, opt_state = optimizer.update(grads, opt_state, network)
    network = eqx.apply_updates(network, updates)

    return network, opt_state, loss, carry_out


optimizer = optax.adam(1e-2)
opt_state = optimizer.init(eqx.filter(network, eqx.is_array))

In [ ]:
BATCH_SIZE = 16
N_EPOCHS = 50
train_losses = []

batched_states = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *coarse_states)
num_chunks = len(target_chunks)

# Pre-align initial states with targets based on UNROLL_BLOCKS
initial_dfs = coarse_dfs[::UNROLL_BLOCKS][:num_chunks]
initial_states = jax.tree_util.tree_map(lambda x: x[::UNROLL_BLOCKS][:num_chunks], batched_states)

pbar = tqdm(range(N_EPOCHS), desc="training")
for epoch in pbar:
    epoch_loss = 0.0

    for i in range(0, num_chunks, BATCH_SIZE):
        targets_batch = target_chunks[i : i + BATCH_SIZE]
        dfs_batch = initial_dfs[i : i + BATCH_SIZE]
        states_batch = jax.tree_util.tree_map(lambda x: x[i : i + BATCH_SIZE], initial_states)

        carry_batch = (dfs_batch, states_batch)

        network, opt_state, batch_loss_val, carry_batch = train_batch(
            network, opt_state, carry_batch, targets_batch
        )

        epoch_loss += float(batch_loss_val) * len(targets_batch)

    epoch_loss /= num_chunks
    train_losses.append(epoch_loss)
    pbar.set_postfix(loss=f"{epoch_loss:.4e}")

## Evaluation

Three-way comparison: fine / coarse / coarse + correction. Show flux traces, $k_y$ spectra, and potential structure.

In [ ]:
df_corr, _, state_corr = gk_init_fn(geometry_coarse, params_coarse)
corr_eflux = []

pbar = tqdm(range(N_BLOCKS))
for b in pbar:
    df_corr, (phi, fluxes), state_corr = gksolve(
        df_corr,
        geometry_coarse,
        params_coarse,
        state_corr,
        n_steps=STEPS_PER_BLOCK,
        pre=pre_coarse,
    )
    ky_energy = jnp.sum(phi.real**2 + phi.imag**2, axis=(0, 1))
    # ky_shift, ky_scale = network(ky_energy)
    # ky_shift = ky_shift[None, None, None, None, :]
    # ky_scale = ky_scale[None, None, None, None, :]
    # df_corr = df_corr * ky_scale + ky_scale
    net_output = network(ky_energy)
    df_corr = df_corr * net_output[None, None, None, None, :]

    _, fluxes_corr = get_integrals(
        df_corr,
        geometry_coarse,
        params=params_coarse,
        pre=pre_coarse,
        adiabatic_electrons=params_coarse.adiabatic_electrons,
    )
    corr_eflux.append(float(fluxes_corr[1]))
    pbar.set_postfix(flux=float(fluxes_corr[1]))

corr_eflux = np.array(corr_eflux)
print(f"corrected eflux range: [{corr_eflux.min():.4f}, {corr_eflux.max():.4f}]")

In [ ]:
# --- paper figure: flux trace comparison ---
fig, ax = plt.subplots(figsize=(7.0, 2.8))
ax.plot(
    fine_times,
    fine_fluxes[:, 1],
    color=JAX_COLORS["blue"],
    label=f"fine ($N_{{ky}}$={len(ky_fine)})",
)
ax.plot(
    coarse_times,
    coarse_fluxes[:, 1],
    color=JAX_COLORS["red"],
    ls="--",
    alpha=0.7,
    label=f"coarse ($N_{{ky}}$={COARSE_NKY})",
)
ax.plot(coarse_times, corr_eflux, color=JAX_COLORS["cyan"], label="coarse + correction")
ax.set_xlabel(r"time $[v_{th}/R]$")
ax.set_ylabel(r"$Q_i$")
ax.legend(frameon=False)
ax.grid(True)
fig.tight_layout()
fig.savefig("figs/correction_fluxes.pdf")
plt.show()

In [ ]:
# --- training loss ---
fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.semilogy(train_losses, color=JAX_COLORS["purple"])
ax.set_xlabel("epoch")
ax.set_ylabel("MSE ($Q_i$)")
ax.grid(True, which="both")
fig.tight_layout()
fig.savefig("figs/correction_loss.pdf")
plt.show()

# --- time-averaged flux comparison ---
avg_start = max(0, N_BLOCKS - N_BLOCKS // 4)
fine_avg = np.mean(fine_fluxes[avg_start:, 1])
coarse_avg = np.mean(coarse_fluxes[avg_start:, 1])
corr_avg = np.mean(corr_eflux[avg_start:])

fig, ax = plt.subplots(figsize=(3.5, 2.8))
bars = ax.bar(
    ["fine", "coarse", "corrected"],
    [fine_avg, coarse_avg, corr_avg],
    color=[JAX_COLORS["blue"], JAX_COLORS["red"], JAX_COLORS["cyan"]],
    alpha=0.85,
)
ax.set_ylabel(r"$\langle Q_i \rangle$")
ax.grid(True, axis="y")
fig.tight_layout()
fig.savefig("figs/correction_bar.pdf")
plt.show()

# MSE summary
mse_coarse = float(np.mean((coarse_fluxes[:, 1] - fine_fluxes[:N_BLOCKS, 1]) ** 2))
mse_corr = float(np.mean((corr_eflux - fine_fluxes[:N_BLOCKS, 1]) ** 2))
print(f"MSE coarse:    {mse_coarse:.4e}")
print(f"MSE corrected: {mse_corr:.4e}")
print(f"improvement:   {mse_coarse / max(mse_corr, 1e-30):.1f}x")
print(f"\ntime-averaged eflux:")
print(f"  fine:      {fine_avg:.4f}")
print(f"  coarse:    {coarse_avg:.4f}")
print(f"  corrected: {corr_avg:.4f}")